In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft as fft
from torch.utils.data import DataLoader, TensorDataset, Subset
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ================== Step 1: Load and Prepare CIFAR-10 Dataset ==================

def load_cifar10_dataset():
    """Load CIFAR-10 dataset with improved augmentation"""
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])
    
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])
    
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                            download=True, transform=transform_train)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                           download=True, transform=transform_test)
    
    classes = ('plane', 'car', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck')
    
    return trainset, testset, classes

# ================== Step 2: FFT Conversion Functions ==================

def spatial_to_frequency(images):
    """
    Convert spatial domain images to frequency domain with enhanced features
    Returns: freq_features (9 channels), freq_phase, freq_complex
    """
    # Apply 2D FFT
    freq_complex = fft.fft2(images, dim=(-2, -1))
    freq_complex = fft.fftshift(freq_complex, dim=(-2, -1))
    
    # Extract magnitude and phase
    freq_magnitude = torch.abs(freq_complex)
    freq_phase = torch.angle(freq_complex)
    
    # Log-scale magnitude for better representation
    eps = torch.mean(freq_magnitude) * 0.01
    freq_magnitude_log = torch.log(freq_magnitude + eps)
    freq_magnitude_normalized = (freq_magnitude_log - freq_magnitude_log.mean()) / (freq_magnitude_log.std() + 1e-8)
    
    # Phase representation using cosine and sine
    phase_cos = torch.cos(freq_phase)
    phase_sin = torch.sin(freq_phase)
    
    # Concatenate to create 9-channel frequency features
    freq_features = torch.cat([freq_magnitude_normalized, phase_cos, phase_sin], dim=1)
    
    return freq_features, freq_phase, freq_complex

def frequency_to_spatial(freq_magnitude, freq_phase):
    """Convert frequency domain back to spatial domain"""
    # Reconstruct complex frequency representation
    freq_magnitude = torch.exp(freq_magnitude)
    freq_complex = freq_magnitude * torch.exp(1j * freq_phase)
    
    # Apply inverse FFT
    freq_complex = fft.ifftshift(freq_complex, dim=(-2, -1))
    spatial_complex = fft.ifft2(freq_complex, dim=(-2, -1))
    spatial_images = torch.real(spatial_complex)
    
    return spatial_images

# ================== Step 3: Frequency Domain Dataset ==================

class FrequencyDomainDataset(torch.utils.data.Dataset):
    """Custom dataset for frequency domain representation"""
    
    def __init__(self, original_dataset):
        self.original_dataset = original_dataset
        
    def __len__(self):
        return len(self.original_dataset)
    
    def __getitem__(self, idx):
        image, label = self.original_dataset[idx]
        freq_features, freq_phase, _ = spatial_to_frequency(image.unsqueeze(0))
        freq_features = freq_features.squeeze(0)
        freq_phase = freq_phase.squeeze(0)
        
        return freq_features, label, freq_phase

# ================== Step 4: CNN Model for Frequency Domain ==================

class FrequencyDomainCNN(nn.Module):
    """ResNet18-based model optimized for frequency domain (9-channel input)"""
    
    def __init__(self, base_model='resnet18', num_classes=10, dropout_rate=0.4):
        super(FrequencyDomainCNN, self).__init__()
        
        if base_model == 'resnet18':
            self.model = models.resnet18(pretrained=True)
            
            # Modify first conv layer for 9-channel input (3 magnitude + 3 phase_cos + 3 phase_sin)
            self.model.conv1 = nn.Conv2d(9, 64, kernel_size=3, stride=1, padding=1, bias=False)
            self.model.maxpool = nn.Identity()  # Remove maxpool for 32x32 images
            
            # Store feature maps for GradCAM++
            self.feature_maps = None
            self.gradients = None
            
            # Modify final classifier
            self.model.fc = nn.Sequential(
                nn.Dropout(dropout_rate),
                nn.Linear(512, 256),
                nn.ReLU(inplace=False),
                nn.Dropout(dropout_rate * 0.5),
                nn.Linear(256, num_classes)
            )
            
            # Register hook on the last conv layer (layer4)
            self.model.layer4.register_forward_hook(self.save_feature_maps)
            self.model.layer4.register_full_backward_hook(self.save_gradients)
            
        else:
            raise ValueError(f"Unsupported model: {base_model}")
        
        self._initialize_weights()
    
    def save_feature_maps(self, module, input, output):
        """Hook to save feature maps"""
        self.feature_maps = output
    
    def save_gradients(self, module, grad_input, grad_output):
        """Hook to save gradients"""
        self.gradients = grad_output[0]
    
    def _initialize_weights(self):
        """Initialize new layers with proper weights"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        return self.model(x)

# ================== Step 5: Training Functions ==================

class EarlyStopping:
    """Early stopping to prevent overfitting"""
    def __init__(self, patience=7, min_delta=0.0, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_state = None
        
    def __call__(self, val_accuracy, model):
        score = val_accuracy
        
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = model.state_dict()
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            self.counter = 0

def train_model(model, train_loader, val_loader, epochs=30, lr=0.001, weight_decay=1e-4):
    """Train the frequency domain CNN model"""
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=lr * 10,
        epochs=epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.3,
        anneal_strategy='cos'
    )
    
    early_stopping = EarlyStopping(patience=15, min_delta=0.1, verbose=True)
    
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []
    
    best_val_accuracy = 0.0
    best_model_state = None
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for i, (freq_images, labels, _) in enumerate(train_pbar):
            freq_images, labels = freq_images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(freq_images)
            loss = criterion(outputs, labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            scheduler.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100 * correct_train / total_train:.2f}%'
            })
        
        avg_train_loss = running_loss / len(train_loader)
        train_accuracy = 100 * correct_train / total_train
        train_losses.append(avg_train_loss)
        train_accuracies.append(train_accuracy)
        
        # Validation
        model.eval()
        running_val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for freq_images, labels, _ in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                freq_images, labels = freq_images.to(device), labels.to(device)
                outputs = model(freq_images)
                loss = criterion(outputs, labels)
                running_val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        avg_val_loss = running_val_loss / len(val_loader)
        val_accuracy = 100 * correct / total
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)
        
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = model.state_dict()
        
        print(f'\nEpoch [{epoch+1}/{epochs}]')
        print(f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%')
        print(f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')
        print(f'Learning Rate: {optimizer.param_groups[0]["lr"]:.6f}\n')
        
        early_stopping(val_accuracy, model)
        if early_stopping.early_stop:
            print("Early stopping triggered!")
            break
    
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nLoaded best model with validation accuracy: {best_val_accuracy:.2f}%")
    
    return train_losses, val_losses, train_accuracies, val_accuracies

# ================== Step 6: GradCAM++ Implementation ==================

class GradCAMPlusPlus:
    """
    GradCAM++ implementation for frequency domain CNN
    Paper: "Grad-CAM++: Improved Visual Explanations for Deep Convolutional Networks"
    """
    
    def __init__(self, model):
        self.model = model
        self.model.eval()
        
    def generate_cam(self, input_tensor, target_class):
        """
        Generate GradCAM++ heatmap for the target class
        
        Args:
            input_tensor: Input frequency domain image [1, 9, H, W]
            target_class: Target class index
            
        Returns:
            cam: GradCAM++ heatmap [H, W]
        """
        # Forward pass
        self.model.zero_grad()
        output = self.model(input_tensor)
        
        # Get the score for target class
        score = output[0, target_class]
        
        # Backward pass
        score.backward(retain_graph=True)
        
        # Get gradients and feature maps
        gradients = self.model.gradients  # [1, C, H, W]
        feature_maps = self.model.feature_maps  # [1, C, H, W]
        
        # Calculate GradCAM++ weights
        # First derivative
        alpha_numer = gradients.pow(2)
        alpha_denom = 2 * gradients.pow(2) + \
                      (feature_maps * gradients.pow(3)).sum(dim=(2, 3), keepdim=True)
        alpha_denom = torch.where(alpha_denom != 0.0, alpha_denom, torch.ones_like(alpha_denom))
        
        alpha = alpha_numer / alpha_denom
        
        # ReLU on gradients
        relu_grad = F.relu(gradients)
        
        # Weights for each channel
        weights = (alpha * relu_grad).sum(dim=(2, 3), keepdim=True)
        
        # Weighted combination of feature maps
        cam = (weights * feature_maps).sum(dim=1, keepdim=True)
        
        # Apply ReLU to cam
        cam = F.relu(cam)
        
        # Normalize to [0, 1]
        cam = cam.squeeze()
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        
        # Resize to input size
        cam = F.interpolate(cam.unsqueeze(0).unsqueeze(0), 
                           size=input_tensor.shape[-2:], 
                           mode='bilinear', 
                           align_corners=False)
        cam = cam.squeeze().cpu().detach()
        
        return cam

def apply_gradcam_and_map_to_spatial(model, freq_image, phase, target_class, original_image):
    """
    Apply GradCAM++ on frequency domain and map back to spatial domain
    
    Args:
        model: Trained frequency domain CNN
        freq_image: Frequency domain image [1, 9, H, W]
        phase: Phase information [3, H, W]
        target_class: Target class for GradCAM++
        original_image: Original spatial domain image [3, H, W]
        
    Returns:
        cam_spatial: CAM in spatial domain
        cam_freq: CAM in frequency domain
        highlighted: Highlighted original image
        original_np: Original image as numpy array
    """
    
    model.eval()
    freq_input = freq_image.clone().detach().to(device)
    freq_input.requires_grad = True
    
    # Generate GradCAM++ heatmap in frequency domain
    gradcam = GradCAMPlusPlus(model)
    cam_freq = gradcam.generate_cam(freq_input, target_class)
    cam_freq_np = cam_freq.numpy()
    
    # Map frequency domain CAM to spatial domain
    # Extract magnitude channels from frequency features
    freq_magnitude = freq_input[:, :3, :, :].squeeze(0).cpu().detach()
    
    # Apply CAM as a mask to magnitude
    cam_freq_tensor = torch.from_numpy(cam_freq_np).unsqueeze(0)
    masked_magnitude = freq_magnitude * cam_freq_tensor
    
    # Ensure phase has correct dimensions
    if phase.dim() == 4:
        phase = phase.squeeze(0)
    
    # Convert back to spatial domain
    spatial_cam = frequency_to_spatial(masked_magnitude.unsqueeze(0), phase.unsqueeze(0))
    spatial_cam = spatial_cam.squeeze(0)
    
    # Create saliency map from spatial CAM
    spatial_cam = torch.abs(spatial_cam)
    saliency_map = torch.mean(spatial_cam, dim=0).numpy()
    
    # Apply Gaussian smoothing if scipy available
    try:
        from scipy.ndimage import gaussian_filter
        saliency_map = gaussian_filter(saliency_map, sigma=1.5)
    except ImportError:
        pass
    
    # Normalize saliency map
    if saliency_map.max() > saliency_map.min():
        saliency_map = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min())
    else:
        saliency_map = np.zeros_like(saliency_map)
    
    # Apply threshold to focus on important regions
    threshold = np.percentile(saliency_map, 60)
    saliency_map = np.where(saliency_map > threshold, saliency_map, 0)
    
    # Re-normalize after thresholding
    if saliency_map.max() > 0:
        saliency_map = (saliency_map - saliency_map.min()) / (saliency_map.max() - saliency_map.min())
    
    # Denormalize original image
    if original_image.dim() == 4:
        original_image = original_image.squeeze(0)
    
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    std = torch.tensor([0.2023, 0.1994, 0.2010]).view(3, 1, 1)
    original_denorm = original_image.cpu() * std + mean
    original_denorm = torch.clamp(original_denorm, 0, 1)
    original_np = original_denorm.permute(1, 2, 0).numpy()
    
    # Create highlighted image
    saliency_colored = plt.cm.jet(saliency_map)[:, :, :3]
    alpha = 0.6 * saliency_map[:, :, np.newaxis]
    highlighted = (1 - alpha) * original_np + alpha * saliency_colored
    highlighted = np.clip(highlighted, 0, 1)
    
    return spatial_cam, cam_freq_np, saliency_map, highlighted, original_np

# ================== Step 7: Visualization Functions ==================

def plot_gradcam_results(original_np, freq_magnitude, cam_freq, saliency_map, 
                        highlighted, prediction, true_label, classes, confidence):
    """Plot comprehensive GradCAM++ analysis results"""
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 11))
    fig.suptitle('Frequency Domain CNN - GradCAM++ Explainability Analysis', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    # Original Image
    axes[0, 0].imshow(original_np)
    axes[0, 0].set_title(f'Original Image\nGround Truth: {classes[true_label]}', 
                         fontsize=12, fontweight='bold')
    axes[0, 0].axis('off')
    
    # Frequency Domain Magnitude
    freq_display = freq_magnitude[:, :3, :, :].squeeze(0).mean(0).cpu().numpy()
    im1 = axes[0, 1].imshow(freq_display, cmap='viridis')
    axes[0, 1].set_title('Frequency Domain\n(Magnitude Spectrum)', 
                         fontsize=12, fontweight='bold')
    axes[0, 1].axis('off')
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)
    
    # Prediction Info
    correct = "✓" if prediction == true_label else "✗"
    color = 'green' if prediction == true_label else 'red'
    axes[0, 2].text(0.5, 0.5, f'{correct} Prediction: {classes[prediction]}\nConfidence: {confidence:.1f}%', 
                    ha='center', va='center', fontsize=14, fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor=color, alpha=0.3))
    axes[0, 2].set_title('Model Prediction', fontsize=12, fontweight='bold')
    axes[0, 2].axis('off')
    
    # GradCAM++ in Frequency Domain
    im2 = axes[1, 0].imshow(freq_display, cmap='gray')
    axes[1, 0].imshow(cam_freq, cmap='jet', alpha=0.5)
    axes[1, 0].set_title('GradCAM++ (Frequency Domain)', 
                        fontsize=12, fontweight='bold')
    axes[1, 0].axis('off')
    
    # Saliency Map in Spatial Domain
    im3 = axes[1, 1].imshow(saliency_map, cmap='hot')
    axes[1, 1].set_title('Saliency Map\n(Mapped to Spatial Domain)', 
                         fontsize=12, fontweight='bold')
    axes[1, 1].axis('off')
    plt.colorbar(im3, ax=axes[1, 1], fraction=0.046, pad=0.04)
    
    # Highlighted Important Regions
    axes[1, 2].imshow(highlighted)
    axes[1, 2].set_title('Highlighted Regions\n(Important for Prediction)', 
                         fontsize=12, fontweight='bold')
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

def plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies):
    """Plot training and validation curves"""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = range(1, len(train_losses) + 1)
    ax1.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
    ax1.plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, train_accuracies, 'b-', label='Training Accuracy', linewidth=2)
    ax2.plot(epochs, val_accuracies, 'r-', label='Validation Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# ================== Step 8: Main Execution Pipeline ==================

def main():
    print("="*80)
    print("Frequency Domain CNN with GradCAM++ Explainability Pipeline")
    print("="*80)
    
    # Step 1: Load CIFAR-10 dataset
    print("\n[Step 1] Loading CIFAR-10 dataset...")
    trainset, testset, classes = load_cifar10_dataset()
    
    # Step 2: Split into train and validation
    print("\n[Step 2] Splitting dataset into train/validation sets...")
    train_indices, val_indices = train_test_split(
        list(range(len(trainset))), 
        test_size=0.15,
        random_state=42,
        stratify=[trainset[i][1] for i in range(len(trainset))]
    )
    
    train_subset = Subset(trainset, train_indices)
    val_subset = Subset(trainset, val_indices)
    
    # Step 3: Convert to frequency domain
    print("\n[Step 3] Converting to frequency domain using FFT...")
    freq_train_dataset = FrequencyDomainDataset(train_subset)
    freq_val_dataset = FrequencyDomainDataset(val_subset)
    freq_test_dataset = FrequencyDomainDataset(testset)
    
    train_loader = DataLoader(freq_train_dataset, batch_size=128, shuffle=True, 
                             num_workers=2, pin_memory=True)
    val_loader = DataLoader(freq_val_dataset, batch_size=128, shuffle=False,
                           num_workers=2, pin_memory=True)
    test_loader = DataLoader(freq_test_dataset, batch_size=1, shuffle=False)
    
    # Step 4: Initialize model
    print("\n[Step 4] Initializing ResNet18 model for frequency domain...")
    model = FrequencyDomainCNN(base_model='resnet18', num_classes=10, dropout_rate=0.4).to(device)
    
    # Step 5: Train model
    print("\n[Step 5] Training model on frequency domain data...")
    train_losses, val_losses, train_accuracies, val_accuracies = train_model(
        model, train_loader, val_loader, 
        epochs=30,
        lr=0.001,
        weight_decay=5e-4
    )
    
    # Plot training curves
    print("\n[Step 5.1] Plotting training curves...")
    plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies)
    
    # Step 6: Evaluate on test set
    print("\n[Step 6] Evaluating on test set...")
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for freq_images, labels, _ in tqdm(test_loader, desc="Testing"):
            freq_images, labels = freq_images.to(device), labels.to(device)
            outputs = model(freq_images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    test_accuracy = 100 * correct / total
    print(f"\nFinal Test Accuracy: {test_accuracy:.2f}%")
    
    # Step 7: Apply GradCAM++ and visualize
    print("\n[Step 7] Applying GradCAM++ and mapping to spatial domain...")
    
    # Test on multiple samples
    test_indices = [0, 10, 20, 30, 40]
    
    for idx in test_indices:
        # Get original image
        original_image, true_label = testset[idx]
        
        # Convert to frequency domain
        freq_features, phase, _ = spatial_to_frequency(original_image.unsqueeze(0))
        freq_input = freq_features.to(device)
        
        # Get prediction
        model.eval()
        with torch.no_grad():
            output = model(freq_input)
            probabilities = F.softmax(output, dim=1)
            confidence, predicted = torch.max(probabilities.data, 1)
            predicted_class = predicted.item()
            confidence = confidence.item() * 100
        
        # Apply GradCAM++ and map to spatial domain
        spatial_cam, cam_freq, saliency_map, highlighted, original_np = apply_gradcam_and_map_to_spatial(
            model, freq_input, phase.squeeze(0), predicted_class, original_image
        )
        
        # Display results
        print(f"\nSample {idx}:")
        print(f"True Label: {classes[true_label]}, Predicted: {classes[predicted_class]} ({confidence:.1f}% confidence)")
        
        plot_gradcam_results(
            original_np,
            freq_features, 
            cam_freq,
            saliency_map, 
            highlighted,
            predicted_class, 
            true_label, 
            classes,
            confidence
        )
        
        # Clear GPU cache
        torch.cuda.empty_cache()
    
    print("\n" + "="*80)
    print("GradCAM++ Pipeline completed successfully!")
    print(f"Final Test Accuracy: {test_accuracy:.2f}%")
    print("="*80)

if __name__ == "__main__":
    main()